# Convert the PDF to Text and Store the text into SQLite3 DB

### Prompt used:
```
You are an expert Python developer. 
Write a clean, production-quality Python script that performs the following task.

Objective:
Read two PDF files (rep1.pdf and rep2.pdf) from a directory, extract their text content, and store the extracted text in a SQLite3 database.

Functional Requirements:
1.	The program should:
 - Open and read the PDF files rep1.pdf and rep2.pdf.
 - Extract all textual content from each file.
 - Store the extracted text in a SQLite database.

2.	The SQLite database should contain a table with the following schema:

Column Name	Description
filename	Name of the PDF file
content	Extracted text content from the PDF

3.	Each row should represent one PDF file and its extracted content.

Technical Requirements
-	Use Python 3.
-	Use sqlite3 for database operations.
-	Use a reliable PDF text extraction library such as PyPDF2 or pdfplumber.

The program should:
-	Automatically create the database if it does not exist.
-	Create the table if it does not already exist.
-	Insert the extracted data into the table.

Code Quality Requirements
•	Write well-structured, modular Python code.
•	Use functions where appropriate.
•	Include clear comments explaining each step.
•	Include basic error handling (e.g., file not found, empty PDFs).

Output
Provide a complete Python script that can be executed directly.
```

This prompt forces the AI to produce better code because it specifies:
•	Role (expert Python developer)
•	Objective
•	Functional requirements
•	Technical stack
•	Code quality expectations
•	Output format
This reduces hallucinations and messy code.



In [1]:
!pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --------------------------- ------------ 4.5/6.6 MB 22.4 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 21.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 22.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 24.5 MB/s  0:00:00

   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------- ----------------------------- 1/4 [cryptography]
   ---------- ----------------------------- 1/4 [cryptography]
   ---------- ----------------------------- 1/4 [cryptography]
   -


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
pdf_to_sqlite.py
----------------
Reads rep1.pdf and rep2.pdf, extracts their text,
and stores each file's content in a SQLite database.

Table schema:
    documents(id INTEGER PRIMARY KEY, filename TEXT, content TEXT)

Requirements:
    pip install pdfplumber
"""

import sqlite3
import pdfplumber
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────

PDF_FILES = ["rep1.pdf", "rep2.pdf"]   # PDFs must be in the same folder
DB_PATH   = "reports.db"               # Output SQLite database

# ── Database setup ───────────────────────────────────────────────────────────

def create_db(db_path: str) -> sqlite3.Connection:
    """Create (or open) the database and ensure the table exists."""
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT NOT NULL,
            content  TEXT
        )
    """)
    conn.commit()
    return conn


# ── PDF text extraction ───────────────────────────────────────────────────────

def extract_text(pdf_path: str) -> str:
    """Extract and return all text from a PDF file."""
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text() or ""
            text_parts.append(f"--- Page {i} ---\n{page_text}")
    return "\n\n".join(text_parts)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    conn = create_db(DB_PATH)

    for pdf_file in PDF_FILES:
        path = Path(pdf_file)

        if not path.exists():
            print(f"  [SKIP] {pdf_file} not found — skipping.")
            continue

        print(f"  [READ] Extracting text from {pdf_file} ...")
        content = extract_text(str(path))

        conn.execute(
            "INSERT INTO documents (filename, content) VALUES (?, ?)",
            (path.name, content)
        )
        conn.commit()
        print(f"  [SAVE] Stored '{path.name}' in '{DB_PATH}'.")

    # ── Verify ────────────────────────────────────────────────────────────────
    print("\n── Records in database ──────────────────────────────")
    for row in conn.execute("SELECT id, filename, length(content) FROM documents"):
        print(f"  id={row[0]}  filename={row[1]}  content_length={row[2]} chars")

    conn.close()
    print(f"\nDone. Database saved to: {Path(DB_PATH).resolve()}")


if __name__ == "__main__":
    main()

  [READ] Extracting text from rep1.pdf ...
  [SAVE] Stored 'rep1.pdf' in 'reports.db'.
  [READ] Extracting text from rep2.pdf ...
  [SAVE] Stored 'rep2.pdf' in 'reports.db'.

── Records in database ──────────────────────────────
  id=1  filename=rep1.pdf  content_length=2695 chars
  id=2  filename=rep2.pdf  content_length=3021 chars

Done. Database saved to: C:\Users\hi\Desktop\projects\python_projects\tutorial\tut_tensorflow\class\reports.db
